# initialize_framework

**Source:** `05_orchestration/initialize_framework.py`  
**Purpose:** Databricks notebook auto-generated from framework Python module.


## Section 1: Additional module code and configuration

This cell handles: *Additional module code and configuration*


In [ ]:
"""Initialize Databricks Unity Catalog infrastructure for the ingestion framework."""

from __future__ import annotations

import sys
from pathlib import Path


## Section 2: Define `_resolve_common_dir()` function with logic for processing

This cell handles: *Define `_resolve_common_dir()` function with logic for processing*


In [ ]:
def _resolve_common_dir() -> Path:
    """Resolve notebooks/00_common path in both local and Databricks runtimes."""
    candidates: list[Path] = []

    # File/module execution path (local, pytest, script execution)
    file_path = globals().get("__file__")
    if file_path:
        current_dir = Path(file_path).resolve().parent
        candidates.extend([
            current_dir.parent / "00_common",
            current_dir / "00_common",
        ])

    # Notebook/runtime cwd fallbacks
    cwd = Path.cwd().resolve()
    candidates.extend([
        cwd / "notebooks" / "00_common",
        cwd / "00_common",
        cwd.parent / "00_common",
    ])

    # Databricks notebook path fallback
    try:
        notebook_path = (
            dbutils.notebook.entry_point.getDbutils()  # type: ignore[name-defined]
            .notebook()
            .getContext()
            .notebookPath()
            .get()
        )
        nb_path = Path(str(notebook_path))
        candidates.extend([
            nb_path.parent.parent / "00_common",
            Path("/Workspace") / nb_path.parent.parent / "00_common",
        ])
    except Exception:
        pass

    # Preserve order, avoid duplicate checks
    seen: set[str] = set()
    for candidate in candidates:
        key = str(candidate)
        if key in seen:
            continue
        seen.add(key)
        if candidate.exists() and candidate.is_dir():
            return candidate

    tried = "\n".join(f"- {p}" for p in candidates)
    raise ModuleNotFoundError(
        "Unable to locate notebooks/00_common for imports. Tried:\n"
        f"{tried}"
    )


COMMON_DIR = _resolve_common_dir()
if str(COMMON_DIR) not in sys.path:
    sys.path.append(str(COMMON_DIR))

try:
    from global_config import load_global_config
    from utils import get_logger
except ModuleNotFoundError:
    # Databricks notebook-only deployments may not expose .py modules on sys.path.
    import json
    import logging
    import os
    import re
    import subprocess
    from datetime import datetime, timezone

    try:
        import yaml
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "PyYAML"])
        import yaml

    _ENV_PATTERN = re.compile(r"\$\{([A-Z0-9_]+)(?::([^}]*))?\}")

    def _substitute_env(value: str) -> str:
        def replacer(match: re.Match[str]) -> str:
            key = match.group(1)
            default_value = match.group(2) if match.group(2) is not None else ""
            return os.getenv(key, default_value)

        return _ENV_PATTERN.sub(replacer, value)

    def _resolve_obj(obj):
        if isinstance(obj, str):
            return _substitute_env(obj)
        if isinstance(obj, dict):
            return {k: _resolve_obj(v) for k, v in obj.items()}
        if isinstance(obj, list):
            return [_resolve_obj(v) for v in obj]
        return obj

    def load_global_config(config_path: str) -> dict:
        path = Path(config_path)
        if not path.exists():
            raise FileNotFoundError(
                f"Global config not found: {path}. "
                "Set --global-config to the correct path or check your deployment."
            )
        with path.open("r", encoding="utf-8") as f:
            payload = yaml.safe_load(f) or {}
        if "lifecycle_log_table" not in payload:
            payload["lifecycle_log_table"] = "audit.entity_lifecycle_log"
        return _resolve_obj(payload)

    class _JsonFormatter(logging.Formatter):
        def format(self, record: logging.LogRecord) -> str:
            payload = {
                "ts": datetime.fromtimestamp(record.created, tz=timezone.utc).isoformat(),
                "level": record.levelname,
                "logger": record.name,
                "message": record.getMessage(),
            }
            if record.exc_info:
                payload["exc_info"] = self.formatException(record.exc_info)
            return json.dumps(payload, default=str)

    def get_logger(name: str = "ingestion_framework") -> logging.Logger:
        logger = logging.getLogger(name)
        if logger.handlers:
            return logger
        logger.setLevel(logging.DEBUG)
        handler = logging.StreamHandler(sys.stdout)
        handler.setFormatter(_JsonFormatter())
        logger.addHandler(handler)
        logger.propagate = False
        return logger

_logger = get_logger("framework_init")


## Section 3: Define `_create_catalog()` function with logic for processing

This cell handles: *Define `_create_catalog()` function with logic for processing*


In [ ]:
def _create_catalog(spark, catalog: str) -> bool:
    """Check if catalog exists, use if available, create if not."""
    try:
        # Check if catalog already exists
        catalogs = spark.sql("SHOW CATALOGS").collect()
        catalog_names = [row["catalog"] for row in catalogs]
        
        if catalog in catalog_names:
            # Catalog exists - get its properties
            try:
                props = spark.sql(f"SHOW CATALOG {catalog}").collect()
                if props:
                    prop_dict = props[0].asDict()
                    _logger.info(f"✓ Catalog '{catalog}' already exists")
                    _logger.info(f"  - Comment: {prop_dict.get('comment', 'N/A')}")
                    _logger.info(f"  - Location: {prop_dict.get('location', 'N/A')}")
                else:
                    _logger.info(f"✓ Catalog '{catalog}' already exists")
            except Exception as e:
                _logger.info(f"✓ Catalog '{catalog}' already exists (properties unavailable: {e})")
            return True
        else:
            # Catalog does not exist - create it
            spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
            _logger.info(f"✓ Catalog '{catalog}' created successfully")
            return True
    except Exception as e:
        _logger.error(f"✗ Failed to initialize catalog '{catalog}': {e}")
        return False


## Section 4: Define `_create_schemas()` function with logic for processing

This cell handles: *Define `_create_schemas()` function with logic for processing*


In [ ]:
def _create_schemas(spark, catalog: str, schemas: dict[str, str], external_location: str = None) -> bool:
    """Create schemas under catalog with optional external location."""
    success = True
    for schema_key, schema_name in schemas.items():
        try:
            if external_location:
                # Create schema with managed location
                spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema_name}")
            else:
                spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema_name}")
            _logger.info(f"✓ Schema '{catalog}.{schema_name}' ready")
        except Exception as e:
            _logger.error(f"✗ Failed to create schema '{catalog}.{schema_name}': {e}")
            success = False
    return success


## Section 5: Define `_build_table_location()` function with logic for processing

This cell handles: *Define `_build_table_location()` function with logic for processing*


In [ ]:
def _build_table_location(external_location: str, schema_name: str, table_name: str) -> str:
    """Build full external location path for table."""
    # Remove trailing slash if present
    base = external_location.rstrip('/')
    return f"{base}/{schema_name}/{table_name}/"


## Section 6: Define `_create_table_with_location()` function with logic for processing

This cell handles: *Define `_create_table_with_location()` function with logic for processing*


In [ ]:
def _create_table_with_location(spark, catalog: str, schema: str, table: str, 
                                ddl: str, external_location: str = None) -> bool:
    """Create table with optional external location."""
    try:
        full_table_name = f"{catalog}.{schema}.{table}"
        if external_location:
            table_path = _build_table_location(external_location, schema, table)
            # Replace the table name in DDL with full three-part name and add LOCATION
            modified_ddl = ddl.replace(
                f"{catalog}.{schema}.{table} (",
                f"{catalog}.{schema}.{table} ("
            )
            # SQL with LOCATION clause
            spark.sql(f"{modified_ddl}\nLOCATION '{table_path}'")
            _logger.info(f"✓ Table '{full_table_name}' created at {table_path}")
        else:
            spark.sql(ddl)
            _logger.info(f"✓ Table '{full_table_name}' ready")
        return True
    except Exception as e:
        _logger.error(f"✗ Failed to create table: {e}")
        return False


## Section 7: Define `_create_control_tables()` function with logic for processing

This cell handles: *Define `_create_control_tables()` function with logic for processing*


In [ ]:
def _create_control_tables(spark, catalog: str, control_schema: str, external_location: str = None) -> bool:
    """Create control metadata tables with optional external location."""
    success = True
    
    # source_registry table
    source_registry_ddl = f"""
        CREATE TABLE IF NOT EXISTS {catalog}.{control_schema}.source_registry (
            tenant STRING,
            brand STRING,
            region STRING,
            product_area STRING,
            product_name STRING,
            source_system STRING,
            source_entity STRING,
            source_type STRING,
            source_format STRING,
            source_path STRING,
            jdbc_profile STRING,
            jdbc_table STRING,
            api_profile STRING,
            api_endpoint STRING,
            api_method STRING,
            api_query_params_json STRING,
            load_type STRING,
            watermark_column STRING,
            landing_table STRING,
            conformance_table STRING,
            silver_table STRING,
            primary_key STRING,
            publish_mode STRING,
            is_active BOOLEAN,
            source_options_json STRING,
            pre_landing_transform_notebook STRING,
            post_conformance_transform_notebook STRING,
            custom_publish_notebook STRING,
            scheduler_name STRING,
            schedule_cron STRING,
            retention_days INT,
            sttm_profile STRING,
            landing_table_type STRING,
            landing_table_path STRING,
            conformance_table_type STRING,
            conformance_table_path STRING,
            silver_table_type STRING,
            silver_table_path STRING
        )
        USING DELTA
        COMMENT 'Source registry - defines all data sources and routing'
    """
    
    if external_location:
        table_path = _build_table_location(external_location, control_schema, "source_registry")
        try:
            spark.sql(f"{source_registry_ddl}\nLOCATION '{table_path}'")
            _logger.info(f"✓ Table '{catalog}.{control_schema}.source_registry' created at {table_path}")
        except Exception as e:
            _logger.error(f"✗ Failed to create source_registry: {e}")
            success = False
    else:
        try:
            spark.sql(source_registry_ddl)
            _logger.info(f"✓ Table '{catalog}.{control_schema}.source_registry' ready")
        except Exception as e:
            _logger.error(f"✗ Failed to create source_registry: {e}")
            success = False

    # column_mapping table
    column_mapping_ddl = f"""
        CREATE TABLE IF NOT EXISTS {catalog}.{control_schema}.column_mapping (
            product_name STRING,
            source_system STRING,
            source_entity STRING,
            source_column STRING,
            conformance_column STRING,
            conformance_data_type STRING,
            column_description STRING,
            is_active BOOLEAN
        )
        USING DELTA
        COMMENT 'Column transformation rules - maps source to conformance columns'
    """
    
    if external_location:
        table_path = _build_table_location(external_location, control_schema, "column_mapping")
        try:
            spark.sql(f"{column_mapping_ddl}\nLOCATION '{table_path}'")
            _logger.info(f"✓ Table '{catalog}.{control_schema}.column_mapping' created at {table_path}")
        except Exception as e:
            _logger.error(f"✗ Failed to create column_mapping: {e}")
            success = False
    else:
        try:
            spark.sql(column_mapping_ddl)
            _logger.info(f"✓ Table '{catalog}.{control_schema}.column_mapping' ready")
        except Exception as e:
            _logger.error(f"✗ Failed to create column_mapping: {e}")
            success = False

    # dq_rules table
    dq_rules_ddl = f"""
        CREATE TABLE IF NOT EXISTS {catalog}.{control_schema}.dq_rules (
            product_name STRING,
            source_system STRING,
            source_entity STRING,
            rule_name STRING,
            rule_type STRING,
            rule_sql STRING,
            column_name STRING,
            expected_value STRING,
            comparator STRING,
            severity STRING,
            is_active BOOLEAN
        )
        USING DELTA
        COMMENT 'Data quality rules - defines DQ checks per source'
    """
    
    if external_location:
        table_path = _build_table_location(external_location, control_schema, "dq_rules")
        try:
            spark.sql(f"{dq_rules_ddl}\nLOCATION '{table_path}'")
            _logger.info(f"✓ Table '{catalog}.{control_schema}.dq_rules' created at {table_path}")
        except Exception as e:
            _logger.error(f"✗ Failed to create dq_rules: {e}")
            success = False
    else:
        try:
            spark.sql(dq_rules_ddl)
            _logger.info(f"✓ Table '{catalog}.{control_schema}.dq_rules' ready")
        except Exception as e:
            _logger.error(f"✗ Failed to create dq_rules: {e}")
            success = False

    # publish_rules table
    publish_rules_ddl = f"""
        CREATE TABLE IF NOT EXISTS {catalog}.{control_schema}.publish_rules (
            product_name STRING,
            source_system STRING,
            source_entity STRING,
            publish_mode STRING,
            merge_key STRING,
            partition_columns_json STRING,
            optimize_zorder_json STRING
        )
        USING DELTA
        COMMENT 'Publish rules - defines silver layer publish behavior'
    """
    
    if external_location:
        table_path = _build_table_location(external_location, control_schema, "publish_rules")
        try:
            spark.sql(f"{publish_rules_ddl}\nLOCATION '{table_path}'")
            _logger.info(f"✓ Table '{catalog}.{control_schema}.publish_rules' created at {table_path}")
        except Exception as e:
            _logger.error(f"✗ Failed to create publish_rules: {e}")
            success = False
    else:
        try:
            spark.sql(publish_rules_ddl)
            _logger.info(f"✓ Table '{catalog}.{control_schema}.publish_rules' ready")
        except Exception as e:
            _logger.error(f"✗ Failed to create publish_rules: {e}")
            success = False

    return success


## Section 8: Define `_create_audit_tables()` function with logic for processing

This cell handles: *Define `_create_audit_tables()` function with logic for processing*


In [ ]:
def _create_audit_tables(spark, catalog: str, audit_schema: str, external_location: str = None) -> bool:
    """Create audit and monitoring tables with optional external location."""
    success = True

    # pipeline_runs table
    pipeline_runs_ddl = f"""
        CREATE TABLE IF NOT EXISTS {catalog}.{audit_schema}.pipeline_runs (
            framework_name STRING,
            load_id STRING,
            source_system STRING,
            source_entity STRING,
            landing_row_count BIGINT,
            valid_row_count BIGINT,
            status STRING,
            error_message STRING,
            run_timestamp TIMESTAMP
        )
        USING DELTA
        COMMENT 'Pipeline execution audit log'
    """
    
    if external_location:
        table_path = _build_table_location(external_location, audit_schema, "pipeline_runs")
        try:
            spark.sql(f"{pipeline_runs_ddl}\nLOCATION '{table_path}'")
            _logger.info(f"✓ Table '{catalog}.{audit_schema}.pipeline_runs' created at {table_path}")
        except Exception as e:
            _logger.error(f"✗ Failed to create pipeline_runs: {e}")
            success = False
    else:
        try:
            spark.sql(pipeline_runs_ddl)
            _logger.info(f"✓ Table '{catalog}.{audit_schema}.pipeline_runs' ready")
        except Exception as e:
            _logger.error(f"✗ Failed to create pipeline_runs: {e}")
            success = False

    # dq_rule_results table
    dq_rule_results_ddl = f"""
        CREATE TABLE IF NOT EXISTS {catalog}.{audit_schema}.dq_rule_results (
            load_id STRING,
            source_system STRING,
            source_entity STRING,
            rule_name STRING,
            failed_row_count BIGINT,
            run_timestamp TIMESTAMP
        )
        USING DELTA
        COMMENT 'DQ rule execution results'
    """
    
    if external_location:
        table_path = _build_table_location(external_location, audit_schema, "dq_rule_results")
        try:
            spark.sql(f"{dq_rule_results_ddl}\nLOCATION '{table_path}'")
            _logger.info(f"✓ Table '{catalog}.{audit_schema}.dq_rule_results' created at {table_path}")
        except Exception as e:
            _logger.error(f"✗ Failed to create dq_rule_results: {e}")
            success = False
    else:
        try:
            spark.sql(dq_rule_results_ddl)
            _logger.info(f"✓ Table '{catalog}.{audit_schema}.dq_rule_results' ready")
        except Exception as e:
            _logger.error(f"✗ Failed to create dq_rule_results: {e}")
            success = False

    # bronze_dq_results table
    bronze_dq_results_ddl = f"""
        CREATE TABLE IF NOT EXISTS {catalog}.{audit_schema}.bronze_dq_results (
            load_id STRING,
            source_system STRING,
            source_entity STRING,
            check_name STRING,
            result STRING,
            run_timestamp TIMESTAMP
        )
        USING DELTA
        COMMENT 'Bronze layer DQ check results'
    """
    
    if external_location:
        table_path = _build_table_location(external_location, audit_schema, "bronze_dq_results")
        try:
            spark.sql(f"{bronze_dq_results_ddl}\nLOCATION '{table_path}'")
            _logger.info(f"✓ Table '{catalog}.{audit_schema}.bronze_dq_results' created at {table_path}")
        except Exception as e:
            _logger.error(f"✗ Failed to create bronze_dq_results: {e}")
            success = False
    else:
        try:
            spark.sql(bronze_dq_results_ddl)
            _logger.info(f"✓ Table '{catalog}.{audit_schema}.bronze_dq_results' ready")
        except Exception as e:
            _logger.error(f"✗ Failed to create bronze_dq_results: {e}")
            success = False

    # rejects table
    rejects_ddl = f"""
        CREATE TABLE IF NOT EXISTS {catalog}.{audit_schema}.rejects (
            load_id STRING,
            source_system STRING,
            source_entity STRING,
            rejected_payload STRING,
            dq_failed_rule STRING,
            run_timestamp TIMESTAMP
        )
        USING DELTA
        COMMENT 'Rejected rows from DQ checks'
    """
    
    if external_location:
        table_path = _build_table_location(external_location, audit_schema, "rejects")
        try:
            spark.sql(f"{rejects_ddl}\nLOCATION '{table_path}'")
            _logger.info(f"✓ Table '{catalog}.{audit_schema}.rejects' created at {table_path}")
        except Exception as e:
            _logger.error(f"✗ Failed to create rejects: {e}")
            success = False
    else:
        try:
            spark.sql(rejects_ddl)
            _logger.info(f"✓ Table '{catalog}.{audit_schema}.rejects' ready")
        except Exception as e:
            _logger.error(f"✗ Failed to create rejects: {e}")
            success = False

    # validation_events table
    validation_events_ddl = f"""
        CREATE TABLE IF NOT EXISTS {catalog}.{audit_schema}.validation_events (
            load_id STRING,
            source_system STRING,
            source_entity STRING,
            event_name STRING,
            event_status STRING,
            event_payload STRING,
            run_timestamp TIMESTAMP
        )
        USING DELTA
        COMMENT 'Validation and execution events'
    """
    
    if external_location:
        table_path = _build_table_location(external_location, audit_schema, "validation_events")
        try:
            spark.sql(f"{validation_events_ddl}\nLOCATION '{table_path}'")
            _logger.info(f"✓ Table '{catalog}.{audit_schema}.validation_events' created at {table_path}")
        except Exception as e:
            _logger.error(f"✗ Failed to create validation_events: {e}")
            success = False
    else:
        try:
            spark.sql(validation_events_ddl)
            _logger.info(f"✓ Table '{catalog}.{audit_schema}.validation_events' ready")
        except Exception as e:
            _logger.error(f"✗ Failed to create validation_events: {e}")
            success = False

    return success


## Section 9: Define `initialize_framework()` function with logic for processing

This cell handles: *Define `initialize_framework()` function with logic for processing*


In [ ]:
def initialize_framework(config_path: str) -> bool:
    """Initialize all UC infrastructure for the framework."""
    try:
        # Import Spark (only available in Databricks)
        from pyspark.sql import SparkSession
        spark = SparkSession.getActiveSession()
        if spark is None:
            _logger.error("✗ No active Spark session. Run this in Databricks only.")
            return False
    except ImportError:
        _logger.error("✗ PySpark not available. Run this in Databricks only.")
        return False

    # Load global config
    try:
        global_config = load_global_config(config_path)
    except Exception as e:
        _logger.error(f"✗ Failed to load config: {e}")
        return False

    # Extract UC config
    db_config = global_config.get("databricks", {})
    catalog = db_config.get("catalog")
    bronze_schema = db_config.get("bronze_schema")
    silver_schema = db_config.get("silver_schema")
    audit_schema = db_config.get("audit_schema")
    external_location = db_config.get("external_location")
    use_external_location = db_config.get("use_external_location", "true").lower() == "true"

    if not all([catalog, bronze_schema, silver_schema, audit_schema]):
        _logger.error("✗ Missing required databricks config: catalog, bronze_schema, silver_schema, audit_schema")
        return False

    _logger.info(f"Initializing framework for catalog={catalog}")
    if use_external_location and external_location:
        _logger.info(f"Using external location: {external_location}")

    # Create catalog
    if not _create_catalog(spark, catalog):
        return False

    # Create schemas
    schemas = {
        "bronze": bronze_schema,
        "silver": silver_schema,
        "audit": audit_schema,
        "control": db_config.get("control_schema", "control"),
    }
    ext_loc = external_location if use_external_location else None
    if not _create_schemas(spark, catalog, schemas, ext_loc):
        return False

    # Create control tables
    control_schema = db_config.get("control_schema", "control")
    if not _create_control_tables(spark, catalog, control_schema, ext_loc):
        return False

    # Create audit tables
    if not _create_audit_tables(spark, catalog, audit_schema, ext_loc):
        return False

    _logger.info(f"✓ Framework initialization complete for {catalog}")
    return True


## Section 10: Define `_resolve_default_global_config_path()` function with logic for processing

This cell handles: *Define `_resolve_default_global_config_path()` function with logic for processing*


In [ ]:
def _resolve_default_global_config_path() -> str:
    """Resolve config/global_config.yaml across script and notebook runtimes."""
    candidates: list[Path] = []

    # Script/module execution path
    file_path = globals().get("__file__")
    if file_path:
        base = Path(file_path).resolve()
        candidates.extend([
            base.parents[2] / "config" / "global_config.yaml",
            base.parent / "config" / "global_config.yaml",
        ])

    # Notebook/local cwd fallbacks
    cwd = Path.cwd().resolve()
    candidates.extend([
        cwd / "config" / "global_config.yaml",
        cwd.parent / "config" / "global_config.yaml",
        cwd.parent.parent / "config" / "global_config.yaml",
    ])

    # Databricks workspace notebook path fallback
    try:
        notebook_path = (
            dbutils.notebook.entry_point.getDbutils()  # type: ignore[name-defined]
            .notebook()
            .getContext()
            .notebookPath()
            .get()
        )
        nb_path = Path(str(notebook_path))
        workspace_base = Path("/Workspace") / nb_path.parent.parent.parent
        candidates.append(workspace_base / "config" / "global_config.yaml")
        # Bundle sync stores repository files under <root>/files/**
        candidates.append(workspace_base / "files" / "config" / "global_config.yaml")
    except Exception:
        pass

    seen: set[str] = set()
    for candidate in candidates:
        key = str(candidate)
        if key in seen:
            continue
        seen.add(key)
        if candidate.exists() and candidate.is_file():
            return str(candidate)

    # Keep a sensible default even if not found; downstream validation gives clear error.
    return str((Path.cwd().resolve() / "config" / "global_config.yaml"))


## Section 11: Define `_resolve_global_config_from_widget()` function with logic for processing

This cell handles: *Define `_resolve_global_config_from_widget()` function with logic for processing*


In [ ]:
def _resolve_global_config_from_widget() -> str | None:
    """Resolve --global-config passed through Databricks notebook task base parameters."""
    try:
        dbutils.widgets.text("global-config", "")  # type: ignore[name-defined]
        widget_value = dbutils.widgets.get("global-config")  # type: ignore[name-defined]
        if widget_value and str(widget_value).strip():
            return str(widget_value).strip()
    except Exception:
        pass
    return None


## Section 12: Define `main()` function with logic for processing

This cell handles: *Define `main()` function with logic for processing*


In [ ]:
def main() -> int:
    """Entry point for initialization. Returns process exit code."""
    import argparse

    parser = argparse.ArgumentParser(description="Initialize Databricks UC infrastructure for ingestion framework")
    parser.add_argument("--global-config", default=None)
    # Notebook kernels inject extra argv values; ignore unknown args there.
    args, _unknown = parser.parse_known_args()

    config_path = (
        args.global_config
        or _resolve_global_config_from_widget()
        or _resolve_default_global_config_path()
    )

    success = initialize_framework(config_path)
    return 0 if success else 1


if __name__ == "__main__":
    in_notebook = "ipykernel" in sys.modules and "__file__" not in globals()
    exit_code = main()
    if not in_notebook:
        raise SystemExit(exit_code)
